<a href="https://colab.research.google.com/github/Dana-El/CCI-HW5/blob/main/Lab3_LangGraph_Workflows_Student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 3: LangGraph — Stateful Workflows
## CCI Session 5

**Duration:** 15 minutes

### Clinical Scenario
> Build a clinical triage system: a patient presents with symptoms, the system classifies severity (low/medium/high), routes to appropriate handling (self-care advice, schedule appointment, or emergency alert), and produces a structured output.

### Objective
- Build a `StateGraph` with typed state
- Create nodes that read and update state
- Add conditional routing based on state
- Visualize the workflow graph

---
## Setup

In [1]:
!pip install -q langchain langchain-openai langgraph grandalf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.4/120.4 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 1.9 MB/s eta 0:00:00


In [3]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
print("Setup complete.")

Setup complete.


---
## Cell 1 — Define the State Schema

In LangGraph, the **state** is a typed dictionary that flows through the graph. Each node can read from and write to the state.

Think of it as the patient's chart that gets passed from station to station in the triage process.

In [4]:
from typing import TypedDict, Literal, Annotated
from langgraph.graph import add_messages

class TriageState(TypedDict):
    patient_symptoms: str
    severity: str
    recommendation: str
    messages: Annotated[list, add_messages]

print("State schema defined.")
print("Fields:", list(TriageState.__annotations__.keys()))

State schema defined.
Fields: ['patient_symptoms', 'severity', 'recommendation', 'messages']


---
## Cell 2 — Define the Nodes

Each node is a function that:
1. Receives the current state
2. Does some work (may call an LLM)
3. Returns a dict with the state keys it wants to update

Our triage workflow has 5 nodes:
- **intake_node**: Records the patient's symptoms
- **classify_node**: Uses LLM to classify severity
- **low_severity_node**: Provides self-care advice
- **medium_severity_node**: Schedules an appointment
- **high_severity_node**: Triggers emergency alert

In [5]:
from langchain_core.messages import SystemMessage, HumanMessage
from pydantic import BaseModel, Field
from typing import Literal

# Node 1: Intake — records symptoms into state
def intake_node(state: TriageState) -> dict:
    """Process patient intake — record symptoms."""
    print("[INTAKE] Processing patient symptoms...")
    return {
        "messages": [
            SystemMessage(content="You are a clinical triage nurse at KHCC."),
            HumanMessage(content=f"Patient presents with: {state['patient_symptoms']}")
        ]
    }

class SeverityClassification(BaseModel):
    """Classification of patient symptom severity."""
    severity: Literal["low", "medium", "high"] = Field(
        description="Severity level: low=self-care, medium=appointment needed, high=emergency"
    )
    reasoning: str = Field(description="Brief clinical reasoning for the classification")

def classify_node(state: TriageState) -> dict:
    """Classify symptom severity using LLM."""
    print("[CLASSIFY] Analyzing symptom severity...")
    classifier = llm.with_structured_output(SeverityClassification)
    result = classifier.invoke(f"Classify the severity of the following symptoms: {state['patient_symptoms']}")
    return {"severity": result.severity}

# Node 3: Low severity — self-care advice
def low_severity_node(state: TriageState) -> dict:
    """Handle low severity — provide self-care advice."""
    print("[LOW] Generating self-care advice...")
    response = llm.invoke(
        f"As a triage nurse, provide brief self-care advice for: {state['patient_symptoms']}. "
        f"Keep it to 2-3 sentences. Mention when to seek further care."
    )
    return {"recommendation": f"SELF-CARE: {response.content}"}

# Node 4: Medium severity — schedule appointment
def medium_severity_node(state: TriageState) -> dict:
    """Handle medium severity — schedule appointment."""
    print("[MEDIUM] Scheduling appointment...")
    response = llm.invoke(
        f"As a triage nurse, provide appointment scheduling advice for: {state['patient_symptoms']}. "
        f"Keep it to 2-3 sentences."
    )
    return {"recommendation": f"APPOINTMENT: {response.content}"}

# Node 5: High severity — emergency alert
def high_severity_node(state: TriageState) -> dict:
    """Handle high severity — emergency alert."""
    print("[HIGH] EMERGENCY ALERT!")
    response = llm.invoke(
        f"As a triage nurse, provide emergency instructions for: {state['patient_symptoms']}. "
        f"Keep it to 2-3 sentences. Urge immediate medical attention."
    )
    return {"recommendation": f"EMERGENCY: {response.content}"}

print("All nodes defined.")

All nodes defined.


---
## Cell 3 — Define Conditional Edges

After classification, the graph needs to route to the correct handler based on severity. This is done with a **conditional edge**.

In [6]:
def route_by_severity(state: TriageState) -> str:
    """Route to the appropriate handler based on severity."""
    severity = state.get("severity")
    if severity == "low":
        return "low_handler"
    elif severity == "medium":
        return "medium_handler"
    else:
        return "high_handler"

print("Routing function defined.")

Routing function defined.


---
## Cell 4 — Build and Compile the Graph

Now we assemble the nodes and edges into a `StateGraph`.

In [7]:
from langgraph.graph import StateGraph, START, END

graph = StateGraph(TriageState)
graph.add_node("intake", intake_node)
graph.add_node("classify", classify_node)
graph.add_node("low_handler", low_severity_node)
graph.add_node("medium_handler", medium_severity_node)
graph.add_node("high_handler", high_severity_node)

graph.add_edge(START, "intake")
graph.add_edge("intake", "classify")
graph.add_conditional_edges("classify", route_by_severity)
graph.add_edge("low_handler", END)
graph.add_edge("medium_handler", END)
graph.add_edge("high_handler", END)

triage_app = graph.compile()

print("Graph compiled successfully!")

Graph compiled successfully!


---
## Cell 5 — Run with Different Patient Scenarios

Let's test our triage system with patients of different severity levels.

In [8]:
# Test scenarios
scenarios = [
    {
        "name": "Scenario 1: Mild symptoms",
        "symptoms": "Mild headache for 2 days, no fever, no vision changes. Patient is otherwise healthy."
    },
    {
        "name": "Scenario 2: Moderate symptoms",
        "symptoms": "Persistent cough for 2 weeks, low-grade fever (37.8C), unintentional weight loss of 3kg. Former smoker."
    },
    {
        "name": "Scenario 3: Severe symptoms",
        "symptoms": "Sudden severe chest pain, difficulty breathing, sweating, pain radiating to left arm. History of hypertension."
    }
]

for scenario in scenarios:
    print("\n" + "=" * 60)
    print(f"  {scenario['name']}")
    print("=" * 60)
    print(f"Symptoms: {scenario['symptoms'][:80]}...")
    print()

    result = triage_app.invoke({
        "patient_symptoms": scenario["symptoms"],
        "severity": "",
        "recommendation": "",
        "messages": []
    })

    print(f"\nSeverity: {result['severity'].upper()}")
    print(f"Recommendation: {result['recommendation'][:300]}")


  Scenario 1: Mild symptoms
Symptoms: Mild headache for 2 days, no fever, no vision changes. Patient is otherwise heal...

[INTAKE] Processing patient symptoms...
[CLASSIFY] Analyzing symptom severity...
[LOW] Generating self-care advice...

Severity: LOW
Recommendation: SELF-CARE: For your mild headache, try resting in a quiet, dark room, staying hydrated, and taking over-the-counter pain relievers like ibuprofen or acetaminophen as directed. If the headache worsens, becomes severe, or is accompanied by new symptoms such as nausea, vomiting, or neurological changes

  Scenario 2: Moderate symptoms
Symptoms: Persistent cough for 2 weeks, low-grade fever (37.8C), unintentional weight loss...

[INTAKE] Processing patient symptoms...
[CLASSIFY] Analyzing symptom severity...
[MEDIUM] Scheduling appointment...

Severity: MEDIUM
Recommendation: APPOINTMENT: Given the persistent cough, low-grade fever, and unintentional weight loss, it is important for the patient to be evaluated by a health

---
## Cell 6 — Visualize the Graph

LangGraph can generate a visual representation of your workflow.

In [9]:
# Visualize the graph as ASCII
print("Graph structure:")
print()
triage_app.get_graph().print_ascii()

# You can also get a Mermaid diagram for rendering
print("\n\nMermaid diagram (paste into mermaid.live):")
print(triage_app.get_graph().draw_mermaid())

Graph structure:

+-----------+  
| __start__ |  
+-----------+  
      *        
      *        
      *        
  +--------+   
  | intake |   
  +--------+   
      *        
      *        
      *        
+----------+   
| classify |   
+----------+   
      *        
      *        
      *        
 +---------+   
 | __end__ |   
 +---------+   


Mermaid diagram (paste into mermaid.live):
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__(<p>__start__</p>)
	intake(intake)
	classify(classify)
	low_handler(low_handler)
	medium_handler(medium_handler)
	high_handler(high_handler)
	__end__(<p>__end__</p>)
	__start__ --> intake;
	intake --> classify;
	classify --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



---
## Stretch Challenge

1. Add a **follow-up node** after each handler that generates a structured JSON summary:
   ```python
   class TriageSummary(BaseModel):
       patient_symptoms: str
       severity: str
       recommendation: str
       follow_up_days: int
       red_flags_to_watch: list[str]
   ```
2. Add **memory** using `MemorySaver` so the triage system can remember previous patient encounters:
   ```python
   from langgraph.checkpoint.memory import MemorySaver
   memory = MemorySaver()
   triage_app = graph.compile(checkpointer=memory)
   ```
3. Run the same patient through twice and see if the system references the prior visit.

### KHCC Connection
This triage workflow pattern is directly applicable at KHCC:
- Emergency department triage for oncology patients
- Phone triage for patients calling with symptoms between treatments
- Chemotherapy side-effect severity assessment
- Routing patients to the right clinic (medical oncology, radiation, surgical, palliative)
- The graph can be extended with more nodes for lab ordering, specialist referral, etc.